# Notebook 5 — Teammate Driver Rating Model

This notebook estimates Formula 1 driver performance using teammate-normalized comparisons instead of raw finish prediction.

Core idea: compare each driver to their teammate in the same race, then link those comparisons across seasons, circuits, and teams using a driver fixed-effects model.

Primary output: `teammate_driver_rating`, a centered driver fixed-effect rating.


## Cell 1 — Setup


In [ ]:
# ==============================================================================
# CELL 1 — SETUP
# ==============================================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = PROJECT_ROOT / "data"
PROCESSED = DATA / "processed"
OUTPUTS = DATA / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

INPUT_FILE = PROCESSED / "driver_race_with_circuit_features.csv"

model = pd.read_csv(INPUT_FILE)

# Notebook 4 may contain duplicated context columns from prior merges.
# Standardize names so formula terms work consistently.
context_column_map = {
    "season_progress_x": "season_progress",
    "season_phase_x": "season_phase",
    "regulation_era_x": "regulation_era",
    "season_total_rounds_x": "season_total_rounds",
}

for old_col, new_col in context_column_map.items():
    if old_col in model.columns and new_col not in model.columns:
        model[new_col] = model[old_col]

columns_to_drop = [
    "season_progress_y",
    "season_phase_y",
    "regulation_era_y",
    "season_total_rounds_y",
    "circuit_name_metadata",
    "circuit_country_metadata",
    "circuit_locality_metadata",
]
model = model.drop(columns=[c for c in columns_to_drop if c in model.columns])

print("Input file:", INPUT_FILE)
print(f"Rows loaded:    {len(model):,}")
print(f"Columns loaded: {model.shape[1]:,}")
print(f"Seasons:        {model['season'].min()}–{model['season'].max()}")
print(f"Races:          {model['race_id'].nunique():,}")
print(f"Drivers:        {model['driver_id'].nunique():,}")
print(f"Constructors:   {model['constructor_id'].nunique():,}")


## Cell 2 — Prepare Teammate Modeling Data


In [ ]:
# ==============================================================================
# CELL 2 — PREPARE TEAMMATE MODELING DATA
# ==============================================================================

required_columns = [
    "driver_race_id",
    "driver_id",
    "driver_name",
    "constructor_id",
    "constructor_name",
    "race_id",
    "season",
    "round",
    "race_name",
    "teammate_id",
    "teammate_name",
    "finish_position",
    "teammate_finish",
    "finish_gap",
    "beat_teammate",
]

missing_required = [c for c in required_columns if c not in model.columns]
if missing_required:
    raise ValueError(f"Missing required teammate columns: {missing_required}")

team_model = model.copy()

# Use upstream teammate features from Notebook 4. Do not rebuild them here.
if "single_car_entry" in team_model.columns:
    team_model = team_model[team_model["single_car_entry"] != 1].copy()

team_model = team_model.dropna(subset=required_columns).copy()
team_model = team_model.sort_values(["season", "round", "constructor_id", "driver_id"]).reset_index(drop=True)

# Normalize typing for model stability.
for col in ["season", "round", "finish_position", "teammate_finish", "finish_gap", "beat_teammate"]:
    team_model[col] = pd.to_numeric(team_model[col], errors="coerce")

team_model = team_model.dropna(subset=["finish_position", "teammate_finish", "finish_gap", "beat_teammate"]).copy()
team_model["beat_teammate"] = team_model["beat_teammate"].astype(int)
team_model["season"] = team_model["season"].astype(int)
team_model["round"] = team_model["round"].astype(int)

single_car_count = int(team_model["single_car_entry"].sum()) if "single_car_entry" in team_model.columns else 0

print(f"Rows retained:       {len(team_model):,}")
print(f"Drivers:             {team_model['driver_id'].nunique():,}")
print(f"Constructors:        {team_model['constructor_id'].nunique():,}")
print(f"Races:               {team_model['race_id'].nunique():,}")
print(f"Single-car entries:  {single_car_count:,}")


## Cell 3 — Validate Teammate Dataset


In [ ]:
# ==============================================================================
# CELL 3 — VALIDATE TEAMMATE DATASET
# ==============================================================================

validation = {
    "Rows": len(team_model),
    "Drivers": team_model["driver_id"].nunique(),
    "Constructors": team_model["constructor_id"].nunique(),
    "Races": team_model["race_id"].nunique(),
    "Duplicate driver_race_id": int(team_model["driver_race_id"].duplicated().sum()),
    "Missing teammate_id": int(team_model["teammate_id"].isna().sum()),
    "Missing teammate_finish": int(team_model["teammate_finish"].isna().sum()),
    "Missing finish_gap": int(team_model["finish_gap"].isna().sum()),
    "Missing beat_teammate": int(team_model["beat_teammate"].isna().sum()),
}

for key, value in validation.items():
    print(f"{key}: {value:,}")

paired_check = (
    team_model.groupby(["race_id", "constructor_id"])["driver_id"]
    .nunique()
    .value_counts()
    .sort_index()
)

print("\nTeam size distribution after filtering:")
print(paired_check)

failures = []
if validation["Duplicate driver_race_id"] != 0:
    failures.append("duplicate driver_race_id rows")
if validation["Missing teammate_id"] != 0:
    failures.append("missing teammate_id")
if validation["Missing teammate_finish"] != 0:
    failures.append("missing teammate_finish")
if validation["Missing finish_gap"] != 0:
    failures.append("missing finish_gap")
if validation["Missing beat_teammate"] != 0:
    failures.append("missing beat_teammate")

print("\nVERDICT:", "PASS" if not failures else f"FAIL — {failures}")


## Cell 4 — Engineer Composite Target


In [ ]:
# ==============================================================================
# CELL 4 — ENGINEER COMPOSITE TARGET
# ==============================================================================

BEAT_WEIGHT = 0.35
GAP_WEIGHT = 0.65
GAP_CAP = 12

if not np.isclose(BEAT_WEIGHT + GAP_WEIGHT, 1.0):
    raise ValueError("BEAT_WEIGHT and GAP_WEIGHT must sum to 1.0")

team_model["finish_gap_capped"] = team_model["finish_gap"].clip(-GAP_CAP, GAP_CAP)

finish_gap_component = team_model["finish_gap_capped"] - team_model["finish_gap_capped"].mean()
finish_gap_component = finish_gap_component / finish_gap_component.std()

win_component = team_model["beat_teammate"] - team_model["beat_teammate"].mean()
win_component = win_component / win_component.std()

team_model["teammate_performance_score"] = (
    BEAT_WEIGHT * win_component
    + GAP_WEIGHT * finish_gap_component
)
team_model["teammate_performance_score"] -= team_model["teammate_performance_score"].mean()

print(f"Beat/win component weight: {BEAT_WEIGHT:.2f}")
print(f"Finish-gap component weight: {GAP_WEIGHT:.2f}")
print(f"Finish-gap cap: {GAP_CAP}")
print("\nTarget distribution:")
print(team_model["teammate_performance_score"].describe())


## Cell 5 — Validate Composite Target


In [ ]:
# ==============================================================================
# CELL 5 — VALIDATE COMPOSITE TARGET
# ==============================================================================

print("Target missing values:", int(team_model["teammate_performance_score"].isna().sum()))
print("Target mean:", team_model["teammate_performance_score"].mean())
print("Target std:", team_model["teammate_performance_score"].std())

print("\nBeat teammate distribution:")
print(team_model["beat_teammate"].value_counts().sort_index())

print("\nAverage target by beat_teammate:")
print(team_model.groupby("beat_teammate")["teammate_performance_score"].agg(["count", "mean", "median", "std"]))

print("\nTarget by finish-gap bucket:")
gap_bucket = pd.cut(
    team_model["finish_gap_capped"],
    bins=[-12, -6, -3, -1, 0, 1, 3, 6, 12],
    include_lowest=True,
)
print(team_model.groupby(gap_bucket, observed=False)["teammate_performance_score"].mean())

verdict = (
    team_model["teammate_performance_score"].notna().all()
    and abs(team_model["teammate_performance_score"].mean()) < 1e-10
)
print("\nVERDICT:", "PASS" if verdict else "CHECK")


## Cell 6 — Fit Fixed-Effects Teammate Model


In [ ]:
# ==============================================================================
# CELL 6 — FIT FIXED-EFFECTS TEAMMATE MODEL
# ==============================================================================

model_columns = [
    "teammate_performance_score",
    "driver_id",
    "season",
    "race_id",
    "circuit_type",
    "season_phase",
    "regulation_era",
    "downforce_score",
    "speed_score",
    "technical_score",
    "overtaking_score",
    "lap_length_km",
    "altitude_m",
    "crash_dnf",
]

missing_model_cols = [c for c in model_columns if c not in team_model.columns]
if missing_model_cols:
    raise ValueError(f"Missing model columns: {missing_model_cols}")

model_data = team_model.dropna(subset=model_columns).copy()

# Keep categorical variables as strings/categories so statsmodels handles them cleanly.
for col in ["driver_id", "season", "circuit_type", "season_phase", "regulation_era"]:
    model_data[col] = model_data[col].astype(str)

teammate_formula = """
teammate_performance_score ~
C(driver_id)
+ C(season)
+ C(circuit_type)
+ C(season_phase)
+ C(regulation_era)
+ downforce_score
+ speed_score
+ technical_score
+ overtaking_score
+ lap_length_km
+ altitude_m
+ crash_dnf
"""

teammate_model = smf.ols(
    formula=teammate_formula,
    data=model_data,
).fit(
    cov_type="cluster",
    cov_kwds={"groups": model_data["race_id"]},
)

model_data["expected_score"] = teammate_model.fittedvalues
model_data["residual"] = teammate_model.resid

print(f"Rows modeled: {len(model_data):,}")
print(f"Drivers:      {model_data['driver_id'].nunique():,}")
print(f"Races:        {model_data['race_id'].nunique():,}")
print(f"R-squared:    {teammate_model.rsquared:.4f}")
print(f"Adj. R-sq:    {teammate_model.rsquared_adj:.4f}")

print("\nExpected score:")
print(model_data["expected_score"].describe())

print("\nResidual:")
print(model_data["residual"].describe())


## Cell 7 — Validate Fixed-Effects Model


In [ ]:
# ==============================================================================
# CELL 7 — VALIDATE FIXED-EFFECTS MODEL
# ==============================================================================

print("Model observations:", int(teammate_model.nobs))
print("Model parameters:", len(teammate_model.params))
print("Driver terms:", int(teammate_model.params.index.str.contains("C\\(driver_id\\)").sum()))
print("R-squared:", round(teammate_model.rsquared, 4))

print("\nResidual by season:")
print(model_data.assign(season=model_data["season"].astype(int)).groupby("season")["residual"].agg(["count", "mean", "std"]).tail(10))

print("\nResidual by circuit type:")
print(model_data.groupby("circuit_type")["residual"].agg(["count", "mean", "std"]))

verdict = (
    teammate_model.nobs > 0
    and np.isfinite(teammate_model.rsquared)
    and abs(model_data["residual"].mean()) < 1e-8
)
print("\nVERDICT:", "PASS" if verdict else "CHECK")


## Cell 8 — Extract Driver Ratings


In [ ]:
# ==============================================================================
# CELL 8 — EXTRACT DRIVER RATINGS
# ==============================================================================

# Statsmodels omits one reference driver from C(driver_id). Add that driver back with a zero coefficient,
# then center all ratings so the reference category does not affect interpretation.
driver_effects = (
    teammate_model.params
    .filter(like="C(driver_id)")
    .rename("teammate_driver_rating")
    .reset_index()
    .rename(columns={"index": "parameter"})
)

driver_effects["driver_id"] = driver_effects["parameter"].str.extract(r"C\(driver_id\)\[T\.(.*)\]")
driver_effects = driver_effects[["driver_id", "teammate_driver_rating"]]

all_driver_ids = pd.Series(sorted(model_data["driver_id"].unique()), name="driver_id")
missing_driver_ids = sorted(set(all_driver_ids) - set(driver_effects["driver_id"]))
reference_driver = missing_driver_ids[0] if missing_driver_ids else None

if reference_driver is not None:
    driver_effects = pd.concat(
        [
            driver_effects,
            pd.DataFrame({"driver_id": [reference_driver], "teammate_driver_rating": [0.0]}),
        ],
        ignore_index=True,
    )

driver_effects["teammate_driver_rating"] = (
    driver_effects["teammate_driver_rating"]
    - driver_effects["teammate_driver_rating"].mean()
)

driver_summary = (
    model_data.groupby("driver_id")
    .agg(
        driver_name=("driver_name", "last"),
        races=("driver_race_id", "count"),
        seasons=("season", "nunique"),
        constructors=("constructor_id", "nunique"),
        avg_finish=("finish_position", "mean"),
        avg_teammate_finish=("teammate_finish", "mean"),
        teammate_win_rate=("beat_teammate", "mean"),
        avg_finish_gap=("finish_gap", "mean"),
        avg_target=("teammate_performance_score", "mean"),
    )
    .reset_index()
)

if "grid_position" in model_data.columns and "teammate_grid" in model_data.columns:
    quali_summary = (
        model_data.dropna(subset=["grid_position", "teammate_grid"])
        .assign(beat_teammate_quali=lambda d: (d["grid_position"] < d["teammate_grid"]).astype(int))
        .groupby("driver_id")
        .agg(qualifying_win_rate=("beat_teammate_quali", "mean"))
        .reset_index()
    )
    driver_summary = driver_summary.merge(quali_summary, on="driver_id", how="left")
else:
    driver_summary["qualifying_win_rate"] = np.nan

if "points" in model_data.columns and "teammate_points" in model_data.columns:
    points_summary = (
        model_data.assign(points_gap=lambda d: d["points"] - d["teammate_points"])
        .groupby("driver_id")
        .agg(avg_points=("points", "mean"), avg_points_gap=("points_gap", "mean"))
        .reset_index()
    )
    driver_summary = driver_summary.merge(points_summary, on="driver_id", how="left")
else:
    driver_summary["avg_points"] = np.nan
    driver_summary["avg_points_gap"] = np.nan

if "car_performance_score" in model_data.columns:
    car_summary = (
        model_data.groupby("driver_id")
        .agg(avg_car_score=("car_performance_score", "mean"))
        .reset_index()
    )
    driver_summary = driver_summary.merge(car_summary, on="driver_id", how="left")
else:
    driver_summary["avg_car_score"] = np.nan

ratings_all = driver_effects.merge(driver_summary, on="driver_id", how="left")

ratings_all["reliability"] = pd.cut(
    ratings_all["races"],
    bins=[-np.inf, 29, 89, 149, np.inf],
    labels=["Low", "Moderate", "Medium", "High"],
)

ratings_all = ratings_all.sort_values("teammate_driver_rating", ascending=False).reset_index(drop=True)
ratings_all["overall_rank"] = np.arange(1, len(ratings_all) + 1)

MIN_RACES = 30
ratings_qualified = ratings_all[ratings_all["races"] >= MIN_RACES].copy()
ratings_qualified["qualified_rank"] = np.arange(1, len(ratings_qualified) + 1)

print(f"Estimated driver effects: {len(ratings_all):,}")
print(f"Qualified drivers:        {len(ratings_qualified):,}")
print(f"Reference driver:         {reference_driver}")

print("\nRating distribution:")
print(ratings_all["teammate_driver_rating"].describe())

print("\nTop 10 qualified drivers:")
print(ratings_qualified[["qualified_rank", "driver_name", "teammate_driver_rating", "reliability", "races"]].head(10).to_string(index=False))

print("\nBottom 10 qualified drivers:")
print(ratings_qualified[["qualified_rank", "driver_name", "teammate_driver_rating", "reliability", "races"]].tail(10).to_string(index=False))


## Cell 9 — Driver Rating Diagnostics


In [ ]:
# ==============================================================================
# CELL 9 — DRIVER RATING DIAGNOSTICS
# ==============================================================================

print("Reliability distribution:")
print(ratings_all["reliability"].value_counts().sort_index())

print("\nTop qualified drivers with context:")
context_cols = [
    "qualified_rank",
    "driver_name",
    "teammate_driver_rating",
    "reliability",
    "races",
    "teammate_win_rate",
    "avg_finish_gap",
    "qualifying_win_rate",
    "avg_points_gap",
    "avg_car_score",
]
print(ratings_qualified[context_cols].head(15).to_string(index=False))

print("\nLowest qualified drivers with context:")
print(ratings_qualified[context_cols].tail(15).to_string(index=False))

print("\nPotential caution cases: high rating with fewer than 50 races")
caution = ratings_qualified[(ratings_qualified["races"] < 50) & (ratings_qualified["qualified_rank"] <= 20)]
print(caution[context_cols].to_string(index=False))


## Cell 10 — Final Rating Validation


In [ ]:
# ==============================================================================
# CELL 10 — FINAL RATING VALIDATION
# ==============================================================================

validation_cols = [
    "teammate_driver_rating",
    "avg_car_score",
    "teammate_win_rate",
    "qualifying_win_rate",
    "avg_finish_gap",
    "avg_points_gap",
    "avg_points",
]
existing_validation_cols = [c for c in validation_cols if c in ratings_qualified.columns]

corr_table = ratings_qualified[existing_validation_cols].corr(numeric_only=True)

print("Qualified drivers:", len(ratings_qualified))
print("\nCorrelation checks:")
print(corr_table.round(3))

car_corr = corr_table.loc["teammate_driver_rating", "avg_car_score"] if "avg_car_score" in corr_table.columns else np.nan
win_corr = corr_table.loc["teammate_driver_rating", "teammate_win_rate"] if "teammate_win_rate" in corr_table.columns else np.nan
finish_gap_corr = corr_table.loc["teammate_driver_rating", "avg_finish_gap"] if "avg_finish_gap" in corr_table.columns else np.nan

print("\nKey validation metrics:")
print(f"Rating vs avg car score:      {car_corr:.3f}" if pd.notna(car_corr) else "Rating vs avg car score:      n/a")
print(f"Rating vs teammate win rate:  {win_corr:.3f}" if pd.notna(win_corr) else "Rating vs teammate win rate:  n/a")
print(f"Rating vs avg finish gap:     {finish_gap_corr:.3f}" if pd.notna(finish_gap_corr) else "Rating vs avg finish gap:     n/a")

if pd.notna(car_corr):
    if abs(car_corr) < 0.20:
        car_verdict = "PASS — low car-score relationship"
    elif abs(car_corr) < 0.35:
        car_verdict = "CHECK — moderate car-score relationship"
    else:
        car_verdict = "FAIL — rating may still be car-contaminated"
else:
    car_verdict = "CHECK — no car score available"

print("\nCar-contamination verdict:", car_verdict)


## Cell 11 — Predictive Validation


In [ ]:
# ==============================================================================
# CELL 11 — PREDICTIVE VALIDATION
# ==============================================================================

prediction_data = model_data.merge(
    ratings_all[["driver_id", "teammate_driver_rating"]],
    on="driver_id",
    how="left",
)

# Simple directional check: does the higher-rated teammate tend to win the intra-team matchup?
pair_rating = prediction_data[[
    "driver_race_id", "race_id", "constructor_id", "driver_id", "teammate_id", "beat_teammate", "teammate_driver_rating"
]].copy()

teammate_lookup = ratings_all[["driver_id", "teammate_driver_rating"]].rename(
    columns={"driver_id": "teammate_id", "teammate_driver_rating": "teammate_rating"}
)
pair_rating = pair_rating.merge(teammate_lookup, on="teammate_id", how="left")
pair_rating["rating_gap"] = pair_rating["teammate_driver_rating"] - pair_rating["teammate_rating"]
pair_rating = pair_rating.dropna(subset=["rating_gap", "beat_teammate"])

pair_rating["higher_rated_driver"] = (pair_rating["rating_gap"] > 0).astype(int)
pair_rating["higher_rated_driver_won"] = (
    (pair_rating["higher_rated_driver"] == 1) & (pair_rating["beat_teammate"] == 1)
) | (
    (pair_rating["higher_rated_driver"] == 0) & (pair_rating["beat_teammate"] == 0)
)

non_ties = pair_rating[pair_rating["rating_gap"].abs() > 1e-12].copy()
accuracy = non_ties["higher_rated_driver_won"].mean()

print(f"Rows checked: {len(non_ties):,}")
print(f"Directional accuracy: {accuracy:.3f}")

print("\nWin rate by rating-gap bucket:")
non_ties["rating_gap_bucket"] = pd.qcut(non_ties["rating_gap"], q=10, duplicates="drop")
print(non_ties.groupby("rating_gap_bucket", observed=False)["beat_teammate"].agg(["count", "mean"]))

print("\nNote: this is an in-sample sanity check, not a production forecasting evaluation.")


## Cell 12 — Model Diagnostics


In [ ]:
# ==============================================================================
# CELL 12 — MODEL DIAGNOSTICS
# ==============================================================================

print("Largest positive residuals:")
print(
    model_data.sort_values("residual", ascending=False)[[
        "season", "round", "race_name", "driver_name", "constructor_name",
        "teammate_name", "finish_position", "teammate_finish", "finish_gap",
        "beat_teammate", "teammate_performance_score", "expected_score", "residual"
    ]].head(15).to_string(index=False)
)

print("\nLargest negative residuals:")
print(
    model_data.sort_values("residual", ascending=True)[[
        "season", "round", "race_name", "driver_name", "constructor_name",
        "teammate_name", "finish_position", "teammate_finish", "finish_gap",
        "beat_teammate", "teammate_performance_score", "expected_score", "residual"
    ]].head(15).to_string(index=False)
)

print("\nResidual summary by driver reliability:")
residual_reliability = model_data.merge(
    ratings_all[["driver_id", "reliability"]],
    on="driver_id",
    how="left",
)
print(residual_reliability.groupby("reliability", observed=False)["residual"].agg(["count", "mean", "std"]))


## Cell 13 — Robustness Snapshot


In [ ]:
# ==============================================================================
# CELL 13 — ROBUSTNESS SNAPSHOT
# ==============================================================================

def fit_rating_variant(input_data, gap_cap=12, beat_weight=0.35, gap_weight=0.65, min_races=30):
    """Fit a compact driver-only variant for sensitivity checks."""
    d = input_data.copy()
    d["finish_gap_capped_variant"] = d["finish_gap"].clip(-gap_cap, gap_cap)
    gap_component = d["finish_gap_capped_variant"] - d["finish_gap_capped_variant"].mean()
    gap_component = gap_component / gap_component.std()
    beat_component = d["beat_teammate"] - d["beat_teammate"].mean()
    beat_component = beat_component / beat_component.std()
    d["target_variant"] = beat_weight * beat_component + gap_weight * gap_component
    d["target_variant"] -= d["target_variant"].mean()

    variant = smf.ols("target_variant ~ C(driver_id) + C(season)", data=d.assign(season=d["season"].astype(str))).fit()

    effects = (
        variant.params
        .filter(like="C(driver_id)")
        .rename("rating_variant")
        .reset_index()
        .rename(columns={"index": "parameter"})
    )
    effects["driver_id"] = effects["parameter"].str.extract(r"C\(driver_id\)\[T\.(.*)\]")
    effects = effects[["driver_id", "rating_variant"]]
    missing = sorted(set(d["driver_id"].astype(str).unique()) - set(effects["driver_id"]))
    if missing:
        effects = pd.concat(
            [effects, pd.DataFrame({"driver_id": [missing[0]], "rating_variant": [0.0]})],
            ignore_index=True,
        )
    effects["rating_variant"] -= effects["rating_variant"].mean()

    counts = d.groupby("driver_id").size().rename("races").reset_index()
    effects = effects.merge(counts, on="driver_id", how="left")
    effects = effects[effects["races"] >= min_races].copy()
    return effects

# Gap cap sensitivity: compare variants to the main qualified ratings.
gap_cap_rows = []
for cap in [8, 10, 12, 15, 20]:
    variant = fit_rating_variant(model_data, gap_cap=cap, min_races=MIN_RACES)
    compare = ratings_qualified[["driver_id", "teammate_driver_rating"]].merge(variant, on="driver_id", how="inner")
    gap_cap_rows.append({
        "gap_cap": cap,
        "drivers_compared": len(compare),
        "corr_with_main_rating": compare[["teammate_driver_rating", "rating_variant"]].corr().iloc[0, 1],
    })

gap_cap_sensitivity = pd.DataFrame(gap_cap_rows)

# Minimum race threshold sensitivity.
threshold_rows = []
for threshold in [10, 20, 30, 50, 75, 100, 150]:
    subset = ratings_all[ratings_all["races"] >= threshold].copy()
    threshold_rows.append({
        "min_races": threshold,
        "qualified_drivers": len(subset),
        "top_driver": subset.iloc[0]["driver_name"] if len(subset) else None,
        "top_rating": subset.iloc[0]["teammate_driver_rating"] if len(subset) else np.nan,
        "rating_std": subset["teammate_driver_rating"].std() if len(subset) > 1 else np.nan,
    })

min_race_threshold_sensitivity = pd.DataFrame(threshold_rows)

print("Gap cap sensitivity:")
print(gap_cap_sensitivity.to_string(index=False))

print("\nMinimum race threshold sensitivity:")
print(min_race_threshold_sensitivity.to_string(index=False))


## Cell 14 — Export Outputs


In [ ]:
# ==============================================================================
# CELL 14 — EXPORT OUTPUTS
# ==============================================================================

ratings_all.to_csv(OUTPUTS / "driver_teammate_ratings_all.csv", index=False)
ratings_qualified.to_csv(OUTPUTS / "driver_teammate_ratings_30plus.csv", index=False)
model_data.to_csv(OUTPUTS / "teammate_model_dataset.csv", index=False)
gap_cap_sensitivity.to_csv(OUTPUTS / "teammate_gap_cap_sensitivity.csv", index=False)
min_race_threshold_sensitivity.to_csv(OUTPUTS / "teammate_min_race_threshold_sensitivity.csv", index=False)

print("Exported files:")
for file_name in [
    "driver_teammate_ratings_all.csv",
    "driver_teammate_ratings_30plus.csv",
    "teammate_model_dataset.csv",
    "teammate_gap_cap_sensitivity.csv",
    "teammate_min_race_threshold_sensitivity.csv",
]:
    path = OUTPUTS / file_name
    print(f"- {path}")

print("\nFinal notebook verdict: ready to run after Notebook 4 output exists.")
